# تحويل أوزان Zazo من Hugging Face إلى حزمة Android محلية

يحوّل هذا الدفتر محوّل LoRA المنشور في `zazo300/zazo-gemma-lora` مع النموذج الأساسي `google/gemma-2b-it` إلى ملفي MediaPipe اللذين يقرأهما تطبيق Zazo: `base.bin` و`lora.bin`. لا يعيد تدريب الأوزان ولا يغيّر المستودع.

**المتطلبات:** شغّل الدفتر في Google Colab مع Runtime من نوع GPU، ثم سجّل الدخول إلى Hugging Face بحساب وافق على شروط Gemma. تنزيل النموذج الأساسي يحتاج عدة غيغابايت، والحزمة الناتجة كبيرة.

بعد إنشاء `zazo_android_model.zip`، نزّله إلى هاتف Android وافتح تطبيق Zazo Brain، ثم اختر **تثبيت حزمة أوزان Android ZIP** وحدد الملف.


In [ ]:
# تثبيت الأدوات المطلوبة
%pip -q install --upgrade mediapipe huggingface_hub


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
from huggingface_hub import hf_hub_download, snapshot_download
from mediapipe.tasks.python.genai import converter
import getpass, hashlib, json, shutil, zipfile

import torch
if not torch.cuda.is_available():
    raise RuntimeError("في Colab اختر Runtime → Change runtime type → GPU ثم أعد تشغيل الدفتر.")

# الإعدادات مثبتة على مستودع الأوزان الذي حدده المستخدم.
ADAPTER_REPO = "zazo300/zazo-gemma-lora"
ADAPTER_CONFIG_FILE = "adapter_config (1).json"
ADAPTER_WEIGHTS_FILE = "adapter_model (1).safetensors"
BASE_MODEL = "google/gemma-2b-it"
LORA_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj"}
DRIVE_ROOT = Path("/content/drive/MyDrive/zazo_brain_lora")

drive.mount("/content/drive", force_remount=False)
token = getpass.getpass("Hugging Face token (يبقى في ذاكرة الجلسة فقط): ").strip()
if not token:
    raise RuntimeError("يلزم Hugging Face token للوصول إلى نموذج Gemma الأساسي. لا تحفظ الرمز في الدفتر.")

output_dir = DRIVE_ROOT / "android_exports" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir.mkdir(parents=True, exist_ok=False)
cache_dir = Path("/content/zazo_hf_cache")
print("مجلد الناتج:", output_dir)


In [ ]:
# تنزيل الإعداد والأوزان، ثم فحص التوافق قبل بدء التحويل.
adapter_config_path = hf_hub_download(ADAPTER_REPO, ADAPTER_CONFIG_FILE, cache_dir=str(cache_dir))
adapter_weights_path = hf_hub_download(ADAPTER_REPO, ADAPTER_WEIGHTS_FILE, cache_dir=str(cache_dir))
adapter_config = json.loads(Path(adapter_config_path).read_text(encoding="utf-8"))
if adapter_config.get("base_model_name_or_path") != BASE_MODEL:
    raise RuntimeError(f"النموذج الأساسي لا يطابق المتوقع: {adapter_config.get('base_model_name_or_path')}")
if int(adapter_config.get("r", -1)) != 8:
    raise RuntimeError(f"رتبة LoRA غير متوقعة: {adapter_config.get('r')}")
if set(adapter_config.get("target_modules", [])) != LORA_TARGETS:
    raise RuntimeError(f"طبقات LoRA غير مدعومة أو غير متوقعة: {adapter_config.get('target_modules')}")
if adapter_config.get("peft_type") != "LORA":
    raise RuntimeError(f"نوع المحوّل غير مدعوم: {adapter_config.get('peft_type')}")
print("فحص التوافق ناجح؛ رتبة LoRA:", adapter_config["r"], "والطبقات:", sorted(LORA_TARGETS))
print(f"حجم محوّل LoRA: {Path(adapter_weights_path).stat().st_size / (1024**2):.1f} MiB")


In [ ]:
# تنزيل Gemma الأساسي المرخّص من Hugging Face.
base_model_dir = cache_dir / "gemma-2b-it"
snapshot_download(
    repo_id=BASE_MODEL,
    local_dir=str(base_model_dir),
    token=token,
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],
)
print("تم تنزيل ملفات النموذج الأساسي إلى:", base_model_dir)


In [ ]:
# التحويل إلى FlatBuffers المدعومة من تطبيق Android.
base_output = output_dir / "zazo_base_gpu.bin"
lora_output = output_dir / "zazo_lora_gpu.bin"
intermediate_dir = output_dir / "intermediate"
conversion_config = converter.ConversionConfig(
    input_ckpt=str(base_model_dir),
    ckpt_format="safetensors",
    model_type="GEMMA_2B",
    backend="gpu",
    output_dir=str(intermediate_dir),
    combine_file_only=False,
    vocab_model_file=str(base_model_dir),
    output_tflite_file=str(base_output),
    lora_ckpt=str(adapter_weights_path),
    lora_rank=int(adapter_config["r"]),
    lora_output_tflite_file=str(lora_output),
)
converter.convert_checkpoint(conversion_config)
if not base_output.is_file() or not lora_output.is_file():
    raise RuntimeError("لم ينشئ MediaPipe ملفي base وLoRA؛ راجع سجل التحويل قبل التثبيت.")
print("اكتمل التحويل:", base_output.stat().st_size, "bytes base;", lora_output.stat().st_size, "bytes LoRA")


In [ ]:
# إنشاء ZIP متوافق مع مثبت التطبيق، مع manifest وبصمات SHA-256.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "format": "mediapipe-llm-inference-gemma-2b-lora",
    "base_model": BASE_MODEL,
    "adapter_repo": ADAPTER_REPO,
    "adapter_revision": "main",
    "lora_rank": int(adapter_config["r"]),
    "target_modules": sorted(LORA_TARGETS),
    "backend": "gpu",
    "base_sha256": sha256_file(base_output),
    "lora_sha256": sha256_file(lora_output),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
manifest_path = output_dir / "android_model_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
zip_path = output_dir / "zazo_android_model.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    archive.write(base_output, "base.bin")
    archive.write(lora_output, "lora.bin")
    archive.write(manifest_path, "android_model_manifest.json")
print("حزمة Android جاهزة:", zip_path)
print(f"حجم الحزمة: {zip_path.stat().st_size / (1024**3):.2f} GiB")




## التثبيت داخل Zazo Brain

1. انتظر اكتمال تنزيل `zazo_android_model.zip` إلى الهاتف.
2. افتح Zazo Brain، ثم اضغط **تثبيت حزمة أوزان Android ZIP** في القائمة الجانبية.
3. اختر ملف ZIP. يستخرج التطبيق `base.bin` و`lora.bin` ويثبت الاثنين معاً.
4. بعد ظهور إشعار نجاح التثبيت، أرسل رسالة في المحادثة؛ سيستخدم Android النموذج المحلي، مع بقاء مسار Worker السحابي احتياطياً.

الأوزان المحلية تحتاج مساحة تخزين كافية وهاتفاً يدعم متطلبات MediaPipe/GPU؛ إذا تعذّر تشغيلها، تبقى المحادثة السحابية الحالية متاحة. لا تختَر ملف `.safetensors` مباشرة في التطبيق، لأنه ليس ملف تشغيل Android.

الملف محفوظ في Google Drive داخل مجلد `zazo_brain_lora/android_exports/`. افتح Google Drive على هاتف Android ونزّل ZIP محلياً، أو اختره من موفّر Drive داخل منتقي الملفات في التطبيق.
